# Phase 1 Session + Mock 验证

这个 notebook 用来验证 Phase 1 后半段新增的最小链路：

- Node 服务能正常启动
- `/api/bootstrap` 能返回语言表和内容目录
- `/api/sessions` 能创建最小 Session 骨架
- mock 模式能生成一段开场文本


In [ ]:
# 这个代码块用于准备路径、端口和基础工具函数，后面的所有验证都会复用它们。
from pathlib import Path
import json
import os
import subprocess
import time
import urllib.request

repo_root = Path(r"f:\Documents\GitHub\AI_TRPG_616\version 3.0")
server_port = 4317
server_base_url = f"http://127.0.0.1:{server_port}"
server_process = None

def http_get_json(url: str) -> dict:
    # 这个辅助函数用于 GET 一个 JSON 接口。
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))

def http_post_json(url: str, payload: dict) -> dict:
    # 这个辅助函数用于 POST 一个 JSON 接口。
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))

print("项目目录:", repo_root)
print("测试端口:", server_port)


In [ ]:
# 这个代码块会启动 Phase 1 的 Node 服务，并等待健康检查通过。
env = os.environ.copy()
env["PORT"] = str(server_port)

server_process = subprocess.Popen(
    ["node", "--experimental-strip-types", ".\\apps\\server\\src\\server.ts"],
    cwd=repo_root,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
)

health = None
for _ in range(20):
    try:
        health = http_get_json(f"{server_base_url}/api/health")
        break
    except Exception:
        time.sleep(0.5)

assert health is not None, "服务启动失败，健康检查没有通过"
print("健康检查结果:", health)


In [ ]:
# 这个代码块验证 bootstrap 接口，确认语言表和内容目录都能被前端初始化页读取。
bootstrap = http_get_json(f"{server_base_url}/api/bootstrap")

print("语言总数:", len(bootstrap["languages"]))
print("规则总数:", len(bootstrap["catalog"]))
print("默认值:", bootstrap["defaults"])

assert len(bootstrap["languages"]) >= 40
assert len(bootstrap["catalog"]) >= 1
assert bootstrap["defaults"]["modelAccessMode"] == "mock"


In [ ]:
# 这个代码块创建一个 mock session，并检查返回的 Session / messages / replay 是否完整。
payload = {
    "ruleDirectoryName": "VHS",
    "storyDirectoryName": "The_Silence",
    "locale": "zh-CN",
    "playMode": "single_player",
    "gmArchitecture": "single_agent",
    "modelAccessMode": "mock",
    "debugEnabled": True,
    "promptDebugEnabled": False,
    "logViewMode": "compact",
}

snapshot = http_post_json(f"{server_base_url}/api/sessions", payload)

print("Session ID:", snapshot["session"]["id"])
print("故事 ID:", snapshot["session"]["storyId"])
print("消息数:", len(snapshot["messages"]))
print("回放数:", len(snapshot["replay"]))

assert snapshot["session"]["status"] == "active"
assert snapshot["session"]["storyId"] == "THE_SILENCE"
assert len(snapshot["messages"]) >= 2
assert len(snapshot["replay"]) >= 3


In [ ]:
# 这个代码块再次读取刚创建的 session，并展示 mock 开场文本。
session_id = snapshot["session"]["id"]
fetched_snapshot = http_get_json(f"{server_base_url}/api/sessions/{session_id}")
opening_message = next(item for item in fetched_snapshot["messages"] if item["kind"] == "gm_narration")

print("请求语言:", fetched_snapshot["contentSummary"]["requestedLocale"])
print("实际语言:", fetched_snapshot["contentSummary"]["resolvedLocale"])
print("\nMock 开场文本:\n")
print(opening_message["content"])

assert "Mock" in opening_message["content"] or "Mock 主持" in opening_message["content"]


In [ ]:
# 这个代码块用于清理后台 Node 进程，建议在验证结束后执行一次。
if server_process is not None and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_process.kill()

print("服务已停止")
